## MLP language model

Motivation: the bigram model in part 1 needed a 27x27 matrix to store probabilities for just 2 characters. 27x27 = 729. 3 characters would need 27x27x27 = 19683. Bengio et al. (2003) propose an alternative model to address this. 

The modeling approach itself is identical to the bigram language model. The difference is that we will now embed each token (character, here) in an n-dimensional vector. This allows some generalizing to novel scenarios by analogy via vector similarity. 

We will use a matrix C to act as a look-up table of {word count}x{feature vector dim} to convert words into vector embeddings.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 
%matplotlib inline

In [ ]:
# read in all the words
#words = open('../names.txt' , 'r').read().splitlines()
words = open('../names.txt', 'r').read().splitlines()
words[:8]

In [ ]:
len(words)

In [ ]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

In [ ]:
# build the dataset

block_size = 3 # context length: how many characters do we use to predict the next one?
X, Y = [], []
for w in words[:5]:
    
    #print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
X.shape, X.dtype, Y.shape, Y.dtype

In [ ]:
# build the dataset

def build_dataset(words):
    block_size = 3 # context length: how many characters do we use to predict the next one?
    X, Y = [], []
    for w in words:
        
        #print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            #print(''.join(itos[i] for i in context), '--->', itos[ix])
            context = context[1:] + [ix] # crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [ ]:
# create the embedding table C
C = torch.randn((27, 2)) # 27 chars; 2-dimensional embeddings
C

In [ ]:
C[5]

# another way to index: treat like a nn layer
# F.one_hot(torch.tensor(5), num_classes=27).float() @ C

`C[X]` replaces every integer in X with its 2-number row from C.

In [ ]:
emb = C[X]
emb.shape

In [ ]:
# construct hidden layer
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [ ]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # -1 lets pytorch infer what the arg should be

In [ ]:
h

In [ ]:
h.shape

In [ ]:
# create the final layer
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [ ]:
logits = h @ W2 + b2

In [ ]:
logits.shape

In [ ]:
counts = logits.exp()

In [ ]:
prob = counts / counts.sum(1, keepdims=True)

In [ ]:
prob.shape

In [ ]:
loss = -prob[torch.arange(X.shape[0]), Y].log().mean()
loss

In [ ]:
# counts = logits.exp()
# prob = counts / counts.sum(1, keepdims=True)
# loss = -prob[torch.arange(X.shape[0), Y].log().mean()
loss = F.cross_entropy(logits, Y) # same loss calculation

### rewrite and cleanup

In [ ]:
X.shape, Y.shape # dataset

In [ ]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 30), generator=g)
W1 = torch.randn((90, 300), generator=g)
b1 = torch.randn(300, generator=g)
W2 = torch.randn((300, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters)

In [ ]:
for p in parameters:
    p.requires_grad = True

In [ ]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [ ]:
lri = []
lossi = []
stepi = []

In [ ]:
for i in range(1000000):
    
    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (96,))
    
    # forward pass (on minibatch)
    emb = C[Xtr[ix]] # (96, 3, 2)
    h = torch.tanh(emb.view(-1, 90) @ W1 + b1) # (96, 100)
    logits = h @ W2 + b2 # (96, 27)
    loss = F.cross_entropy(logits, Ytr[ix])
    #print(loss.item())
    
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # update
    #lr = lrs[i]
    lr = 0.15 if i < 100000 else 0.02 if i < 500000 else 0.01 
    for p in parameters:
        p.data += -lr * p.grad
    
    # track stats
    #lri.append(lre[i])
    stepi.append(i)
    lossi.append(loss.log10().item())
    
#print(loss.item())



In [ ]:
plt.plot(stepi, lossi)

In [ ]:
emb = C[Xtr] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 90) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
emb = C[Xdev] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 90) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

In [ ]:
# visualization for 2-dimensional feature vector
plt.figure(figsize=(8,8))
plt.scatter(C[:,0].data, C[:,1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha="center", va="center", color='white')
plt.grid('minor')

In [ ]:
plt.plot(lri, lossi)

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
        emb = C[torch.tensor([context])] # (1,block_size,d)
        h = torch.tanh(emb.view(1,-1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
        
    print(''.join(itos[i] for i in out))